<div align="center">

  <a href="https://www.ultralytics.com/yolo/yolo27?utm_source=github&utm_medium=social&utm_campaign=yolo27-launch-2026&utm_content=banner" target="_blank">
    <img width="1024", src="https://raw.githubusercontent.com/ultralytics/assets/main/yolov8/banner-yolov8.png"></a>

  [中文](https://docs.ultralytics.com/zh) | [한국어](https://docs.ultralytics.com/ko) | [日本語](https://docs.ultralytics.com/ja) | [Русский](https://docs.ultralytics.com/ru) | [Deutsch](https://docs.ultralytics.com/de) | [Français](https://docs.ultralytics.com/fr) | [Español](https://docs.ultralytics.com/es) | [Português](https://docs.ultralytics.com/pt) | [Türkçe](https://docs.ultralytics.com/tr) | [Tiếng Việt](https://docs.ultralytics.com/vi) | [العربية](https://docs.ultralytics.com/ar)

  <a href="https://github.com/ultralytics/ultralytics/actions/workflows/ci.yml"><img src="https://github.com/ultralytics/ultralytics/actions/workflows/ci.yml/badge.svg" alt="Ultralytics CI"></a>
  <a href="https://clickpy.clickhouse.com/dashboard/ultralytics"><img src="https://static.pepy.tech/badge/ultralytics" alt="Ultralytics Downloads"></a>
  <a href="https://discord.com/invite/ultralytics"><img alt="Ultralytics Discord" src="https://img.shields.io/discord/1089800235347353640?logo=discord&logoColor=white&label=Discord&color=blue"></a>
  <a href="https://community.ultralytics.com"><img alt="Ultralytics Forums" src="https://img.shields.io/discourse/users?server=https%3A%2F%2Fcommunity.ultralytics.com&logo=discourse&label=Forums&color=blue"></a>
  <a href="https://www.reddit.com/r/ultralytics/"><img alt="Ultralytics Reddit" src="https://img.shields.io/reddit/subreddit-subscribers/ultralytics?style=flat&logo=reddit&logoColor=white&label=Reddit&color=blue"></a>
  <br>
  <a href="https://console.paperspace.com/github/ultralytics/notebooks/blob/main/notebooks/how-to-use-ultralytics-yolo-ocr-for-number-plate-recognition.ipynb"><img src="https://assets.paperspace.io/img/gradient-badge.svg" alt="Run Ultralytics on Gradient"></a>
  <a href="https://colab.research.google.com/github/ultralytics/notebooks/blob/main/notebooks/how-to-use-ultralytics-yolo-ocr-for-number-plate-recognition.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Ultralytics In Colab"></a>
  <a href="https://www.kaggle.com/models/ultralytics/yolo26"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open Ultralytics In Kaggle"></a>
  <a href="https://mybinder.org/v2/gh/ultralytics/notebooks/HEAD?labpath=notebooks%2Fhow-to-use-ultralytics-yolo-ocr-for-number-plate-recognition.ipynb"><img src="https://mybinder.org/badge_logo.svg" alt="Open Ultralytics In Binder"></a>

  Welcome to the ANPR with Ultralytics YOLO26 notebook! <a href="https://github.com/ultralytics/ultralytics">YOLO26</a> is the latest version of the YOLO (You Only Look Once) AI models developed by <a href="https://www.ultralytics.com">Ultralytics</a>. We hope that the resources in this notebook will help you get the most out of YOLO26. Please browse the YOLO26 <a href="https://docs.ultralytics.com">Docs</a> for details, raise an issue on <a href="https://github.com/ultralytics/ultralytics">GitHub</a> for support, and join our <a href="https://ultralytics.com/discord">Discord</a> community for questions and discussions!</div>

# Automatic Number Plate Recognition using Ultralytics YOLO26 + OCR

This notebook shows how to build automatic number plate recognition (ANPR) with an Ultralytics YOLO plate detector and the Ultralytics [OCR solution](https://docs.ultralytics.com/guides/ocr), which reads each detected plate with the PP-OCRv6 text recognizer.

## What is Automatic Number Plate Recognition (ANPR)?
Automatic number plate recognition is a technology designed to identify and extract vehicle number plate information from images or videos. Ultralytics YOLO finds the plates, and Ultralytics OCR reads the text on each one, making ANPR an efficient solution for automating vehicle identification tasks.

## Why Use Ultralytics YOLO + OCR for ANPR?

- Accurate Detection: YOLO's object detection capabilities ensure precise localization of license plates in various conditions, such as low light or high-speed movement.

- Local Text Recognition: The OCR solution reads every plate on your own machine with PP-OCRv6 on ONNX Runtime, so there is no API key, no per-request cost, and no image leaves your device.

- Real-Time Processing: Detection and recognition run frame by frame, making it ideal for applications in traffic management, parking systems, and security surveillance.🚗

### Setup

pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

[![PyPI - Version](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![Downloads](https://static.pepy.tech/badge/ultralytics)](https://clickpy.clickhouse.com/dashboard/ultralytics) [![PyPI - Python Version](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)

In [ ]:
!uv pip install ultralytics

import cv2
import ultralytics
from ultralytics import solutions
from ultralytics.utils.downloads import safe_download

ultralytics.checks()

### Download the Video and Model File

Download the sample video we’ll use for processing. If you prefer to use your own video file, downloading the sample is not necessary.  

✅ We’ll download a license plate detection model `anpr-demo-model.pt` trained on a small dataset, designed to detect license plates in the sample video. You are welcome to use your own custom models as well.  

⚠️ Note: This license plate detection model `anpr-demo-model.pt` is intended solely for proof of concept (POC) purposes and may only work with `anpr-demo-video.mp4`.

In [2]:
# download the sample video file
safe_download("https://github.com/ultralytics/assets/releases/download/v0.0.0/anpr-demo-video.mp4")

# download the sample model file
safe_download("https://github.com/ultralytics/assets/releases/download/v0.0.0/anpr-demo-model.pt")

100%|██████████| 9.52M/9.52M [00:00<00:00, 10.4MB/s]


100%|██████████| 5.20M/5.20M [00:00<00:00, 377MB/s]


### Read the Video and Initialize OCR

You can either read the video file directly or stream content from an RTSP (Real-Time Streaming Protocol) source, offering flexible video input options to meet your requirements.

✅ By default, we will use the demo video `anpr-demo-video.mp4` downloaded in the previous step. Additionally, we’ll set up the video writer to manage the output video processing.

✅ The OCR solution loads `anpr-demo-model.pt` to find plates and downloads the PP-OCRv6 recognizer on first use. You can also use your own model for license plate detection.

✅ On an NVIDIA GPU, pass `recognizer="tensorrt"` to read plates with an FP16 TensorRT engine, or pass any callable that takes a BGR crop and returns `(text, confidence)` to use your own recognizer.

In [ ]:
cap = cv2.VideoCapture("anpr-demo-video.mp4")
assert cap.isOpened(), "Error reading video file"

# Video writer
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
video_writer = cv2.VideoWriter("anpr-output.avi", cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

# Detect plates with the license plate model at its 640 training size, then read each plate with PP-OCRv6
ocr = solutions.OCR(model="anpr-demo-model.pt", imgsz=640)

<img align="left" src="https://github.com/user-attachments/assets/101152c3-25ba-48a7-9916-c32c3be5857e" height="640">

### Process Video Frames

In this step, we will process video frames to detect license plates and read their text. Here's how it works:

✅ Frames are read using OpenCV, and the OCR solution detects plates with the YOLO model.

✅ Each detected plate is cropped and read by the PP-OCRv6 recognizer, which returns its text and confidence.

✅ The text is drawn as a label on each plate's bounding box, and the plates read in each frame are printed.

✅ Processed frames are saved to an output video file.

In [ ]:
while cap.isOpened():
    success, im0 = cap.read()

    if not success:
        break

    results = ocr(im0)  # Detect plates and read their text

    print(f"Extracted text: {[t['text'] for t in results.texts]}")

    video_writer.write(results.plot_im)  # Write the processed video frame

cap.release()  # Release the video capture
video_writer.release()  # Release the video writer

<img align="left" src="https://github.com/ultralytics/notebooks/releases/download/v0.0.0/anpr-yolo-ocr-output.jpg" height="640">